In [ ]:
!pip install -q langgraph langchain langchain-core langchain-openai langchain-community chromadb tiktoken
!pip install mcp

In [ ]:
import os
import json
import operator
import asyncio
import re
from typing import TypedDict, Annotated, Sequence, Literal, Optional, List
from datetime import datetime

from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage,
)
from langchain_core.tools import tool
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp.client.session import ClientSession

# Environment Setup

## Stack
- **LLM**: `ChatOpenAI(model="gpt-4o")` — native function-calling for tool selection
- **Orchestration**: LangGraph — cyclic state graphs with conditional edges
- **Vector Store**: ChromaDB with OpenAI embeddings for RAG
- **Observability**: LangSmith — full execution traces

## Why LangGraph?
Traditional chain frameworks (e.g., LangChain's `SequentialChain`) are DAGs — no cycles allowed.
Agents, by definition, need cycles: reason → act → observe → reason again. LangGraph models
execution as a **state machine** where nodes are functions, edges are transitions, and the
state is a typed dictionary that flows through the graph.

```
┌─────────────────────────────────────────────┐
│           LangGraph Architecture             │
│                                             │
│  StateGraph(TypedDict)                      │
│    ├── Node: Python function(state) → dict  │
│    ├── Edge: unconditional transition        │
│    ├── Conditional Edge: route by state      │
│    └── Compile → executable graph            │
│                                             │
│  State flows: node → reducer merges → next  │
└─────────────────────────────────────────────┘
```

## LangSmith Tracing
Set `LANGCHAIN_TRACING_V2=true` to capture every LLM call, tool invocation, and
state transition.

Traces include latency, token counts, and full I/O at each node.
View at https://smith.langchain.com.

In [ ]:
# LangSmith tracing — sign up at https://smith.langchain.com
os.environ["OPENAI_API_KEY"] = ""
os.environ["SERPER_API_KEY"] = ""
os.environ["LANGCHAIN_API_KEY"] = ""

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "agentic-workflow"
llm = ChatOpenAI(model="gpt-4o")

# 1 Agentic Flow

## 1.1 Agent State Design

LangGraph flows a **typed state dictionary** through every node. The key design pattern
is the **reducer** — inspired by Redux — which defines how each field is updated when
multiple nodes write to it.

```
┌──────────────────────────────────────────────┐
│              AgentState (TypedDict)           │
│                                              │
│  messages: Annotated[Sequence, operator.add]  │  ← reducer: APPEND
│  plan: List[str]                             │  ← overwrite
│  step_count: int                             │  ← overwrite
│  max_steps: int                              │  ← overwrite
│  reflection_notes: List[str]                 │  ← overwrite
│  final_answer: Optional[str]                 │  ← overwrite
│  status: str                                 │  ← overwrite
└──────────────────────────────────────────────┘
```

The `messages` field uses `Annotated[Sequence[BaseMessage], operator.add]` which means
when a node returns `{"messages": [new_msg]}`, the new message is **appended** to the
existing list rather than overwriting it.

This is essential for accumulating conversation history across the agent loop.

All other fields use simple overwrite semantics — the latest value wins.


In [ ]:
class AgentState(TypedDict):
    """Central state for the agentic workflow."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    plan: List[str]
    step_count: int
    max_steps: int
    reflection_notes: List[str]
    final_answer: Optional[str]
    status: str  # planning | executing | reflecting | complete | error


def create_initial_state(user_query: str, max_steps: int = 15) -> AgentState:
    """Factory for a properly initialized agent state."""
    return AgentState(
        messages=[HumanMessage(content=user_query)],
        plan=[],
        step_count=0,
        max_steps=max_steps,
        reflection_notes=[],
        final_answer=None,
        status="planning",
    )

# Verify state creation
sample = create_initial_state("test query")
print("State fields:", list(sample.keys()))
print("Messages type:", type(sample["messages"][0]).__name__)

State fields: ['messages', 'plan', 'step_count', 'max_steps', 'reflection_notes', 'final_answer', 'status']
Messages type: HumanMessage


## 1.2 Tool Definitions

Tools give the agent the ability to interact with the external world. LangChain's
`@tool` decorator converts a Python function into a **structured tool** with an
auto-generated JSON schema that the LLM sees during function calling.

```
@tool decorator → JSON Schema → LLM function_call decision → ToolNode executes → ToolMessage
```

### Tool Description Best Practices
1. **Clear name**: The function name becomes the tool name the LLM sees
2. **Detailed docstring**: This becomes the `description` field — be explicit about
   *when* to use the tool and what it returns
3. **Typed arguments**: Python type hints become the JSON schema `properties`
4. **Arg descriptions**: Docstring `Args:` section maps to parameter descriptions

In [ ]:
import os
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool
import re
from datetime import datetime

@tool
def google_search(query: str):
    """
    Use the GoogleSerperAPIWrapper to search Google and return the results.

    Args:
        query (str): The search query to use.

    Returns:
        The search results, as a list of dictionaries.
    """
    search = GoogleSerperAPIWrapper(
        gl="us",  # Country code for search results
        hl="en",  # Language for search results
        type="search",  # Type of search (can be 'search', 'news', 'places', 'images')
        serper_api_key=os.environ.get("SERPER_API_KEY")
    )
    return search.run(query)

search_tool = google_search

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the exact result.
    Use for any computation: arithmetic, percentages, comparisons.

    Args:
        expression: A mathematical expression (e.g. '42 * 17', '5.92 / 5.55').
    """
    try:
        sanitized = re.sub(r'[^0-9+\-*/().,%\s]', '', expression)
        result = eval(sanitized)
        return f"{sanitized.strip()} = {result}"
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"

@tool
def get_current_time() -> str:
    """Get the current date, time, and day of week.
    Use when the user asks about today's date, the current time, or scheduling."""
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S (%A)")

ALL_TOOLS = [search_tool, calculator, get_current_time]
print("Tools registered:", [t.name for t in ALL_TOOLS])

Tools registered: ['google_search', 'calculator', 'get_current_time']


In [ ]:
# Inspect the JSON schema that the LLM actually sees for tool selection.
# This is what gets injected into the function-calling API payload.
for t in ALL_TOOLS:
    schema = t.args_schema.schema() if hasattr(t, 'args_schema') else {}
    print(f"\n--- {t.name} ---")
    print(f"Description: {t.description[:100]}...")
    print(f"Schema: {json.dumps(schema, indent=2)}")


--- google_search ---
Description: Use the GoogleSerperAPIWrapper to search Google and return the results.

Args:
    query (str): The ...
Schema: {
  "description": "Use the GoogleSerperAPIWrapper to search Google and return the results.\n\nArgs:\n    query (str): The search query to use.\n\nReturns:\n    The search results, as a list of dictionaries.",
  "properties": {
    "query": {
      "title": "Query",
      "type": "string"
    }
  },
  "required": [
    "query"
  ],
  "title": "google_search",
  "type": "object"
}

--- calculator ---
Description: Evaluate a mathematical expression and return the exact result.
    Use for any computation: arithme...
Schema: {
  "description": "Evaluate a mathematical expression and return the exact result.\nUse for any computation: arithmetic, percentages, comparisons.\n\nArgs:\n    expression: A mathematical expression (e.g. '42 * 17', '5.92 / 5.55').",
  "properties": {
    "expression": {
      "title": "Expression",
      "type": "string"

/tmp/ipykernel_4262/1276947563.py:4: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  schema = t.args_schema.schema() if hasattr(t, 'args_schema') else {}


## 1.3 RAG with ChromaDB

We expose **Retrieval-Augmented Generation** as a tool so the agent autonomously
decides when to query the knowledge base. This is superior to always-on retrieval
(RAG-as-pre-step) because:

1. **No wasted context**: Irrelevant documents aren't injected for unrelated queries
2. **Agent autonomy**: The LLM learns to route between web search and internal KB
3. **Composability**: RAG is just another tool — no special pipeline changes needed

```
RAG-as-pre-step (wasteful):          RAG-as-tool (agent decides):
                                     
Query ──→ Retrieve ──→ LLM           Query ──→ LLM ──→ [rag_retrieve?] ──→ LLM
          (always)      ↑                              [web_search?]
                        │                              [calculator?]
                    docs injected                      [respond directly?]
```

**Reference**: Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" (NeurIPS 2020).

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

chroma_client = chromadb.Client()

embedding_fn = OpenAIEmbeddingFunction(
    api_key=os.environ.get("OPENAI_API_KEY", ""),
    model_name="text-embedding-3-small",
)

collection = chroma_client.get_or_create_collection(
    name="company_knowledge_base",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

# Synthetic company documents for the knowledge base
documents = [
    "Vacation Policy: Full-time employees receive 20 days PTO per year, accruing at 1.67 days/month. Up to 5 unused days carry over.",
    "Remote Work Policy: Employees may work remotely up to 3 days/week with manager approval. Core hours are 10am-3pm local time.",
    "Expense Reimbursement: Business expenses over $50 require pre-approval. Travel meal cap is $75/day. Submit receipts within 30 days.",
    "Performance Reviews: Semi-annual (Jan and Jul). Self-assessment, peer feedback (2-3 peers), manager evaluation. 1-5 scale.",
    "Onboarding: 2-week program. Week 1: culture, compliance, tooling. Week 2: team-specific technical onboarding with buddy.",
    "Security Policy: Mandatory 2FA. Passwords: min 12 chars, mixed case, numbers, symbols. Rotate every 90 days.",
    "Engineering Standards: PR review required. CI must pass. 80% test coverage threshold. Blue-green deployment with rollback.",
    "Data Retention: Customer PII retained 3 years post-closure. Anonymization after 1 year inactivity. GDPR deletion within 30 days.",
]

collection.upsert(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    metadatas=[{"source": f"handbook_section_{i+1}"} for i in range(len(documents))],
)


@tool
def rag_retrieve(query: str) -> str:
    """Search the internal company knowledge base for policies, procedures,
    and guidelines. Use for questions about PTO, remote work, expenses,
    reviews, onboarding, security, or engineering standards.

    Args:
        query: A natural language question about company policies.
    """
    results = collection.query(query_texts=[query], n_results=3)
    if results and results["documents"] and results["documents"][0]:
        docs = results["documents"][0]
        distances = results["distances"][0] if results.get("distances") else [0] * len(docs)
        formatted = []
        for doc, dist in zip(docs, distances):
            formatted.append(f"[distance: {dist:.3f}] {doc}")
        return "Knowledge Base Results:\n\n" + "\n\n".join(formatted)
    return "No relevant documents found."


ALL_TOOLS_WITH_RAG = ALL_TOOLS + [rag_retrieve]
print(f"ChromaDB: {collection.count()} docs indexed")
print("All tools:", [t.name for t in ALL_TOOLS_WITH_RAG])

ChromaDB: 8 docs indexed
All tools: ['google_search', 'calculator', 'get_current_time', 'rag_retrieve']


## 1.4 Reasoning + Acting

The **ReAct** pattern (Yao et al., ICLR 2023) interleaves LLM reasoning with tool
execution in a cycle. Unlike simple sequential chains, the agent can:
- Call multiple tools in sequence based on intermediate results
- Decide autonomously when it has enough information to respond
- Recover from tool errors by trying alternative approaches

```
┌─────────────────────────────────────────────────────┐
│                  ReAct Agent Loop                    │
│                                                     │
│  START ──→ agent ──→ [has tool_calls?] ──→ tools    │
│              ↑            │ no                ↓     │
│              │            ↓                   │     │
│              │           END                  │     │
│              └────────────────────────────────┘     │
│                                                     │
│  agent node: LLM reasons, may emit tool_calls      │
│  tools node: ToolNode executes, returns results     │
│  conditional edge: inspect tool_calls to route      │
└─────────────────────────────────────────────────────┘
```

### How this differs from sequential chains:
- **Chains**: A → B → C (fixed topology, no cycles)
- **ReAct**: A → B → A → B → A → END (dynamic, agent controls loop count)

The LLM itself decides whether to call a tool (emitting `tool_calls` on the AIMessage)
or respond directly. LangGraph's conditional edge inspects this field to route.

**Reference**: Yao et al., "ReAct: Synergizing Reasoning and Acting in Language Models" (ICLR 2023).

In [ ]:
AGENT_SYSTEM_PROMPT = """You are a senior research assistant with access to external tools.

## Available Tools
1. **web_search** — Search the internet for current facts and statistics.
2. **calculator** — Evaluate mathematical expressions. NEVER do mental math.
3. **get_current_time** — Get today's date and time.
4. **rag_retrieve** — Search internal company knowledge base (policies, procedures).

## Reasoning Protocol
Before acting, think step-by-step:
1. What information do I need?
2. Which tool(s) will provide it?
3. In what order should I call them?

## Rules
- Company policy questions → use rag_retrieve
- Current facts/statistics → use web_search
- Any calculation → use calculator
- NEVER guess at numbers — always verify with a tool
- After gathering info, provide a clear, concise answer"""

def build_react_agent(tools: list):
    """Build a ReAct agent: START -> agent -> [tools | END]."""
    model_with_tools = llm.bind_tools(tools)

    def agent_node(state: AgentState) -> dict:
        """LLM reasons over state and optionally emits tool_calls."""
        system = SystemMessage(content=AGENT_SYSTEM_PROMPT)
        response = model_with_tools.invoke([system] + list(state["messages"]))
        return {
            "messages": [response],
            "step_count": state["step_count"] + 1,
        }

    # execute tool function from msg.tool_calls`
    tool_node = ToolNode(tools)

    def should_continue(state: AgentState) -> Literal["tools", "end"]:
        """Route: tool_calls present -> tools node; otherwise -> END."""
        if state["step_count"] >= state["max_steps"]:
            return "end"
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return "end"

    workflow = StateGraph(AgentState)

    # 2 nodes
    workflow.add_node("agent", agent_node)
    workflow.add_node("tools", tool_node)
    workflow.add_edge(START, "agent")

    # Trigger should_continue after agent node
    # Call tool if exist else END
    workflow.add_conditional_edges(
        "agent", should_continue, {"tools": "tools", "end": END}
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile(checkpointer=MemorySaver())

react_agent = build_react_agent(ALL_TOOLS_WITH_RAG)
print("ReAct Agent compiled. Nodes: agent, tools")

ReAct Agent compiled. Nodes: agent, tools


In [ ]:
def run_agent(agent, query: str, thread_id: str = "default"):
    """Execute the agent and print a compact trace of tool calls and the final answer."""

    print(f"Query: {query}")
    initial = create_initial_state(query)
    config = {"configurable": {"thread_id": thread_id}}
    final_content = ""

    # Stream langraph agent
    for i, event in enumerate(agent.stream(initial, config, stream_mode="updates")):
        print(f"Event {i+1}: {event}")

        for node, output in event.items():
            if "messages" not in output:
                continue

            for msg in output["messages"]:
                if isinstance(msg, AIMessage):

                    if msg.tool_calls:

                        for tc in msg.tool_calls:
                            args_str = json.dumps(tc["args"], ensure_ascii=False)
                            # Decided to call tool, execution will happen next step at tool node
                            print(f"  TOOL [{node}] {tc['name']}({args_str})")
                    elif msg.content:
                        final_content = msg.content

                # returned Tool result
                elif isinstance(msg, ToolMessage):
                    # Here, LangGraph has already executed the tool and yielded the result.
                    preview = msg.content[:120].replace("\n", " ")
                    print(f"  RESULT [{node}] -> {preview}...")

    print(f"\n  ANSWER: {final_content[:500]}")
    print(f"{'='*60}")
    return final_content


# Demo 1: Factual query (web search)
run_agent(react_agent, "What is the population of Singapore?", "react-1")

# Demo 2: Internal knowledge (RAG)
run_agent(react_agent, "What is our company vacation policy?", "react-2")

# Demo 3: Calculation
run_agent(react_agent, "Calculate 1547 * 23 + 891", "react-3")

Query: What is the population of Singapore?
Event 1: {'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 418, 'total_tokens': 438, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f000e9ceb4', 'id': 'chatcmpl-DTTwoUnifD4PbIvagrUEV9GC4aSi1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7d05-7143-7f63-8a67-a03f7bf20fd0-0', tool_calls=[{'name': 'google_search', 'args': {'query': 'current population of Singapore 2023'}, 'id': 'call_5UJVdM2I0MaItaH0ETPTkUJB', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 418, 'output_tokens': 20, 'total_tokens': 438, 'input_token_d

'The result of \\(1547 \\times 23 + 891\\) is 36,472.'

## 1.5 Plan-and-Execute Agents

Pure ReAct agents reason one step at a time, which can fail on complex multi-step
problems. **Plan-and-Execute** separates the workflow into phases:

1. **Planner**: Generate a complete step-by-step plan upfront
2. **Executor**: Handle each step with tool access
3. **Replanner**: Adjust the plan after each step based on new information

```
┌──────────────────────────────────────────────────────────┐
│             Plan-and-Execute Architecture                 │
│                                                          │
│  START ──→ planner ──→ executor ──→ replanner            │
│                          ↑              │                │
│                          │         [more steps?]         │
│                          │          yes │  no            │
│                          └──────────────┘  ↓             │
│                                          END             │
└──────────────────────────────────────────────────────────┘
```

### Why this works better than pure ReAct for complex queries:
- **Global coherence**: The planner sees the full problem before acting
- **Adaptive**: The replanner adjusts based on what the executor discovers
- **Separates concerns**: Planning LLM and execution LLM can have different prompts

In [ ]:
class PlanExecuteState(TypedDict):
    """State for plan-and-execute workflow."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    plan: List[str]
    current_step: int
    step_results: List[str]
    final_answer: Optional[str]
    max_steps: int


def build_plan_execute_agent(tools: list):
    """Build a plan-and-execute agent: planner -> executor -> replanner -> [executor | END]."""
    model_with_tools = llm.bind_tools(tools)
    planner_llm = ChatOpenAI(model="gpt-5.1", temperature=0)
    replanner_llm = ChatOpenAI(model="gpt-5.1", temperature=0)

    def planner_node(state: PlanExecuteState) -> dict:
        """Generate a step-by-step plan from the user query."""
        query = state["messages"][0].content
        response = planner_llm.invoke([
            SystemMessage(content=(
                "You are a planning agent. Given a query, break it into a numbered "
                "list of atomic steps. Each step should be a single action that can "
                "be executed with one tool call. Output ONLY the numbered list."
            )),
            HumanMessage(content=query),
        ])
        steps = [s.strip() for s in response.content.strip().split("\n") if s.strip()]
        plan_msg = f"Plan created with {len(steps)} steps:\n" + "\n".join(steps)
        return {
            "plan": steps,
            "current_step": 0,
            "messages": [AIMessage(content=plan_msg)],
        }

    tool_node = ToolNode(tools)

    def executor_node(state: PlanExecuteState) -> dict:
        """Execute the current plan step using tool-calling LLM."""
        idx = state["current_step"]
        if idx >= len(state["plan"]):
            return {"messages": [AIMessage(content="All plan steps completed.")]}

        step = state["plan"][idx]
        context = ""
        if state["step_results"]:
            context = "Previous results:\n" + "\n".join(state["step_results"][-3:])

        prompt = f"{context}\n\nCurrent step: {step}\n\nExecute this step using the appropriate tool."
        response = model_with_tools.invoke([
            SystemMessage(content="You are an executor agent. Complete the given step using tools."),
            HumanMessage(content=prompt),
        ])
        return {"messages": [response]}

    def executor_postprocess(state: PlanExecuteState) -> dict:
        """Capture executor result and advance step counter."""
        last_msgs = []

        # start from latest message
        for m in reversed(state["messages"]):
            if isinstance(m, (ToolMessage, AIMessage)):
                last_msgs.append(m.content[:200])
                if isinstance(m, AIMessage) and not m.tool_calls:
                    break
            if len(last_msgs) > 3:
                break
        result_summary = " | ".join(reversed(last_msgs))
        new_results = list(state["step_results"]) + [result_summary]

        return {
            "current_step": state["current_step"] + 1,
            "step_results": new_results,
        }

    def replanner_node(state: PlanExecuteState) -> dict:
        """Evaluate progress and decide whether to continue or finish."""

        query = state["messages"][0].content
        results_so_far = "\n".join(state["step_results"])
        remaining = state["plan"][state["current_step"]:]

        response = replanner_llm.invoke([
            SystemMessage(content=(
                "You evaluate plan progress. Given the original query, results so far, and "
                "remaining steps, decide: (a) if the query is fully answered, output 'DONE: <answer>', "
                "or (b) if more steps are needed, output 'CONTINUE'."
            )),
            HumanMessage(content=(
                f"Query: {query}\nResults:\n{results_so_far}\n"
                f"Remaining steps: {remaining}"
            )),
        ])

        content = response.content.strip()
        if content.upper().startswith("DONE"):
            answer = content[5:].strip() if len(content) > 5 else results_so_far
            return {
                "final_answer": answer,
                "messages": [AIMessage(content=answer)],
            }
        return {"messages": [AIMessage(content="Continuing to next step...")]}

    # Routing functions
    def after_executor(state: PlanExecuteState) -> Literal["tools", "executor_post"]:
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return "executor_post"

    def after_replanner(state: PlanExecuteState) -> Literal["executor", "end"]:
        if state.get("final_answer"):
            return "end"
        if state["current_step"] >= len(state["plan"]):
            return "end"
        if state["current_step"] >= state["max_steps"]:
            return "end"
        return "executor"

    # Assemble graph
    wf = StateGraph(PlanExecuteState)
    wf.add_node("planner", planner_node)
    wf.add_node("executor", executor_node)
    wf.add_node("tools", tool_node)
    wf.add_node("executor_post", executor_postprocess)
    wf.add_node("replanner", replanner_node)

    wf.add_edge(START, "planner")
    wf.add_edge("planner", "executor")
    wf.add_conditional_edges(
        "executor", after_executor, {"tools": "tools", "executor_post": "executor_post"}
    )
    wf.add_edge("tools", "executor_post")
    wf.add_edge("executor_post", "replanner")
    wf.add_conditional_edges(
        "replanner", after_replanner, {"executor": "executor", "end": END}
    )

    return wf.compile(checkpointer=MemorySaver())


plan_agent = build_plan_execute_agent(ALL_TOOLS_WITH_RAG)
print("Plan-and-Execute Agent compiled.")

Plan-and-Execute Agent compiled.


In [ ]:
# omplex multi-step query that benefits from planning
def run_plan_agent(query: str, thread_id: str = "plan-demo"):
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    initial = PlanExecuteState(
        messages=[HumanMessage(content=query)],
        plan=[], current_step=0, step_results=[],
        final_answer=None, max_steps=10,
    )
    config = {"configurable": {"thread_id": thread_id}}

    for event in plan_agent.stream(initial, config, stream_mode="updates"):
        for node, output in event.items():

            if "plan" in output and output["plan"]:
                print(f"  PLAN: {len(output['plan'])} steps generated.\nPlan:\n{output['plan']}")

            if "messages" in output:
                for msg in output["messages"]:
                    if isinstance(msg, AIMessage) and msg.tool_calls:
                        for tc in msg.tool_calls:
                            print(f"  TOOL [{node}] {tc['name']}({json.dumps(tc['args'])})")
                    elif isinstance(msg, ToolMessage):
                        print(f"  RESULT [{node}] -> {msg.content[:100]}...")

            if "final_answer" in output and output["final_answer"]:
                print(f"\n  FINAL ANSWER: {output['final_answer'][:500]}")

    print(f"{'='*60}")


run_plan_agent(
    "Compare the populations of Singapore and Norway. "
    "Which is larger and by what percentage?",
    "plan-demo-1",
)


Query: Compare the populations of Singapore and Norway. Which is larger and by what percentage?
  PLAN: 6 steps generated.
Plan:
['1. Look up the most recent reliable population figure for Singapore.', '2. Look up the most recent reliable population figure for Norway.', '3. Confirm that both populations are from the same year or adjust to a common reference year if needed.', '4. Compare the two population values to determine which country has the larger population.', '5. Calculate the percentage difference using the formula: ((larger − smaller) / smaller) × 100.', '6. Report which country has the larger population and the computed percentage difference.']
  TOOL [executor] google_search({"query": "most recent population of Singapore 2023"})
  RESULT [tools] -> 5.918 million (2023)...

  FINAL ANSWER: Using 2023 population estimates:

- Singapore: 5.918 million  
- Norway: 5.51 million  

Singapore’s population is larger.

Percentage difference (taking Norway as the baseline):  
((5.91

## 1.6 Reflection and Self-Critique Loops

A separate **reflection LLM** evaluates the agent's answer against the tool results
and decides whether to ACCEPT or request a REVISION. If revision is needed, the
feedback is injected back into the conversation and the agent retries.

```
┌──────────────────────────────────────────────────────────┐
│            Reflective Agent Architecture                  │
│                                                          │
│  START ──→ agent ──→ [tool_calls?] ──→ tools             │
│              ↑           │ no             ↓              │
│              │           ↓                │              │
│              │       reflector ←──────────┘              │
│              │           │                               │
│              │      [ACCEPT?]                            │
│              │       yes │  no                           │
│              │           ↓   └──── inject feedback ──→↑  │
│              │          END                              │
└──────────────────────────────────────────────────────────┘
```

### Key insight from Reflexion:
The reflection is **verbal** — it's a natural language critique injected as a
HumanMessage. The agent doesn't need gradient updates; it improves by reading
the feedback in context. This is "verbal reinforcement learning."

**Reference**: Shinn et al., "Reflexion: Language Agents with Verbal
Reinforcement Learning" (NeurIPS 2023).

In [ ]:
REFLECTION_SYSTEM_PROMPT = """You are a quality evaluator for an AI research assistant.

## Evaluation Criteria
1. **Accuracy**: Does the answer match the tool results? Are numbers correct?
2. **Completeness**: Does it address ALL parts of the user's question?
3. **Grounding**: Is every claim backed by a tool result (not hallucinated)?
4. **Clarity**: Is the answer well-structured and easy to understand?

## Output Format
Respond with EXACTLY one of:
- "ACCEPT: [brief reason]" — if the answer meets all criteria
- "REVISE: [specific feedback on what to fix]" — if improvements are needed

Be strict. If a number is wrong or a sub-question is unanswered, say REVISE."""


def build_reflective_agent(tools: list, max_reflections: int = 2):
    """ReAct agent with a reflection loop. Agent -> tools -> reflector -> [agent | END]."""
    model_with_tools = llm.bind_tools(tools)
    reflection_llm = ChatOpenAI(model="gpt-4o", temperature=0)

    def agent_node(state: AgentState) -> dict:
        system = SystemMessage(content=AGENT_SYSTEM_PROMPT)
        response = model_with_tools.invoke([system] + list(state["messages"]))
        return {"messages": [response], "step_count": state["step_count"] + 1}

    tool_node = ToolNode(tools)

    def reflector_node(state: AgentState) -> dict:
        """Evaluate the agent's answer. ACCEPT or REVISE with feedback."""
        query = state["messages"][0].content if state["messages"] else ""
        last_answer = ""
        for m in reversed(state["messages"]):
            if isinstance(m, AIMessage) and m.content and not getattr(m, "tool_calls", None):
                last_answer = m.content
                break

        tool_results = [m.content for m in state["messages"] if isinstance(m, ToolMessage)]
        context = (
            f"Original question: {query}\n\n"
            f"Tool results:\n" + "\n".join(f"- {r[:200]}" for r in tool_results[-5:]) +
            f"\n\nAssistant's answer:\n{last_answer}"
        )

        verdict = reflection_llm.invoke([
            SystemMessage(content=REFLECTION_SYSTEM_PROMPT),
            HumanMessage(content=context),
        ])

        note = verdict.content.strip()
        new_notes = list(state["reflection_notes"]) + [note]

        # If REVISE and within budget, inject feedback for the agent
        if "REVISE" in note.upper() and len(new_notes) <= max_reflections:
            return {
                "reflection_notes": new_notes,
                "messages": [HumanMessage(content=f"[REVISION REQUESTED] {note}")],
                "status": "reflecting",
            }

        return {"reflection_notes": new_notes, "status": "complete"}

    # Routing
    def after_agent(state: AgentState) -> Literal["tools", "reflector"]:
        if state["step_count"] >= state["max_steps"]:
            return "reflector"
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return "reflector"

    def after_reflector(state: AgentState) -> Literal["agent", "end"]:
        if state.get("status") == "complete":
            return "end"
        return "agent"

    wf = StateGraph(AgentState)
    wf.add_node("agent", agent_node)
    wf.add_node("tools", tool_node)
    wf.add_node("reflector", reflector_node)

    wf.add_edge(START, "agent")
    wf.add_conditional_edges(
        "agent", after_agent, {"tools": "tools", "reflector": "reflector"}
    )
    wf.add_edge("tools", "agent")
    wf.add_conditional_edges(
        "reflector", after_reflector, {"agent": "agent", "end": END}
    )

    return wf.compile(checkpointer=MemorySaver())


reflective_agent = build_reflective_agent(ALL_TOOLS_WITH_RAG)
print("Reflective Agent compiled. Nodes: agent, tools, reflector")

# === Test ===
run_agent(
    reflective_agent,
    "Is Singapore's population larger than Norway's? By how much, and what is the percentage difference?",
    "reflect-demo",
)

Reflective Agent compiled. Nodes: agent, tools, reflector


## 1.7 Error Handling and Retry Strategies

Production agents must handle failures gracefully. Tools can timeout, APIs can
rate-limit, and LLMs can hallucinate tool calls with bad arguments.

### Key strategies:
1. **Try/except in tools**: Catch and return error messages instead of crashing
2. **Exponential backoff**: For API rate limits
3. **Fallback tools**: Primary tool fails → try backup
4. **LLM self-recovery**: Return the error as a ToolMessage; the agent reads it
   and adjusts its approach autonomously
5. **Max retries**: Prevent infinite loops

```
Tool call ──→ Execute ──→ [Success? return result]
                │ failure
                ↓
            [Retry < max?] ──→ Backoff ──→ Execute again
                │ no
                ↓
            Return error as ToolMessage → LLM adapts
```

In [ ]:
import time
def retry_tool_node(tools: list, max_retries: int = 2, base_delay: float = 1.0):
    """Create a tool execution node with retry logic and error handling."""
    tools_by_name = {t.name: t for t in tools}

    def execute_with_retry(state: AgentState) -> dict:
        results = []
        last_msg = state["messages"][-1]

        if not hasattr(last_msg, "tool_calls") or not last_msg.tool_calls:
            return {"messages": []}

        for tc in last_msg.tool_calls:
            tool_fn = tools_by_name.get(tc["name"])
            if not tool_fn:
                results.append(ToolMessage(
                    content=f"Error: Unknown tool '{tc['name']}'. Available: {list(tools_by_name.keys())}",
                    tool_call_id=tc["id"],
                ))
                continue

            # Retry loop with exponential backoff
            for attempt in range(max_retries + 1):
                try:
                    result = tool_fn.invoke(tc["args"])
                    results.append(ToolMessage(
                        content=str(result),
                        tool_call_id=tc["id"],
                    ))
                    break
                except Exception as e:
                    if attempt < max_retries:
                        delay = base_delay * (2 ** attempt)
                        print(f"  RETRY: {tc['name']} attempt {attempt+1} failed: {e}. "
                              f"Retrying in {delay}s...")
                        time.sleep(delay)
                    else:
                        # Final failure — return error as ToolMessage so LLM can adapt
                        results.append(ToolMessage(
                            content=(
                                f"Error: {tc['name']} failed after {max_retries + 1} attempts. "
                                f"Last error: {e}. Try a different approach."
                            ),
                            tool_call_id=tc["id"],
                        ))

        return {"messages": results}

    return execute_with_retry

# Demo: Build agent with retry-aware tool node
retry_node = retry_tool_node(ALL_TOOLS_WITH_RAG, max_retries=2)
print("Retry-aware tool node created (max_retries=2, exponential backoff)")
print("On failure, error is returned as ToolMessage so the LLM can self-recover.")

Retry-aware tool node created (max_retries=2, exponential backoff)
On failure, error is returned as ToolMessage so the LLM can self-recover.


# 2 Memory Systems

Agentic systems need multiple memory layers, mirroring human cognitive architecture:

```
┌──────────────────────────────────────────────────────────┐
│                   Memory Hierarchy                       │
│                                                          │
│  Short-Term (Context Window)                             │
│    └─ Messages accumulate in state within a single run   │
│    └─ LLM sees full history on every invocation          │
│    └─ Limited by token window (128K for GPT-4o)          │
│                                                          │
│  Long-Term (Checkpointing + Vector Store)                │
│    └─ MemorySaver persists state across invocations      │
│    └─ Same thread_id restores prior conversation         │
│    └─ ChromaDB for semantic retrieval of past episodes   │
│                                                          │
│  Working Memory (Scratchpad)                             │
│    └─ Separate state field for intermediate reasoning    │
│    └─ Agent writes notes, reads them in subsequent steps │
│    └─ Cleared between tasks, persists within a task      │
└──────────────────────────────────────────────────────────┘
```

**Reference**: Park et al., "Generative Agents: Interactive Simulacra of Human Behavior" (2023).

## 2.1 Short-Term Memory (Context Window)

The simplest memory: the `messages` list in `AgentState` grows with every LLM call
and tool result.

The LLM sees the full history on every invocation.

**Limitation**: GPT-4o has a 128K token context window. For long agent runs, the
message history can overflow. Strategies:
- **Truncation**: Keep only the last N messages
- **Summarization**: Periodically summarize older messages
- **Selective**: Keep system prompt + last K turns + all tool results

In [ ]:
import tiktoken

def count_tokens(messages: list, model: str = "gpt-4o") -> int:
    """Count tokens in a message list using tiktoken."""
    enc = tiktoken.encoding_for_model(model)
    total = 0
    for m in messages:
        content = m.content if hasattr(m, "content") else str(m)
        total += len(enc.encode(content)) + 4  # per-message overhead
    return total


def trim_messages(messages: list, max_tokens: int = 100_000) -> list:
    """Keep system + recent messages within token budget."""
    system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
    other_msgs = [m for m in messages if not isinstance(m, SystemMessage)]

    # Always keep system messages; trim from the front of other messages
    while count_tokens(system_msgs + other_msgs) > max_tokens and other_msgs:
        other_msgs.pop(0)

    return system_msgs + other_msgs


# === Test ===
sample_msgs = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is the meaning of life?"),
    AIMessage(content="The meaning of life is a philosophical question..."),
]
print(f"Token count for sample conversation: {count_tokens(sample_msgs)}")
print(f"GPT-4o context window: 128,000 tokens")

Token count for sample conversation: 34
GPT-4o context window: 128,000 tokens


## 2.2 Long-Term Memory (Checkpointing + Vector Store)

LangGraph's **MemorySaver** persists the full graph state after each node execution.
Using the same `thread_id` across invocations restores the prior state — enabling
multi-turn conversations with full history.

For production, replace `MemorySaver` (in-memory) with:
- `SqliteSaver` — file-based persistence
- `PostgresSaver` — production-grade, supports concurrent access

ChromaDB provides **episodic memory** — the agent can retrieve past interactions
by semantic similarity, even across different threads.

In [ ]:
# Demo: Multi-turn conversation with checkpoint memory
THREAD_ID = "memory-long-term-demo"

# Turn 1
run_agent(react_agent, "What is the population of Norway?", THREAD_ID)

# Inspect the checkpoint — state is persisted after the run
config = {"configurable": {"thread_id": THREAD_ID}}
state = react_agent.get_state(config)

print(f"\nCheckpoint: {len(state.values.get('messages', []))} messages persisted")

Query: What is the population of Norway?
Event 1: {'agent': {'messages': [AIMessage(content='The population of Norway is approximately 5.66 million as of October 2023.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 974, 'total_tokens': 993, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a7c3cc1f7d', 'id': 'chatcmpl-DTU67Li3VqRUD9DB3EIn2LjslfyoN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7d0e-41f1-7d70-966f-40d54f109e0a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 974, 'output_tokens': 19, 'total_tokens': 993, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'r

## 2.3 Working Memory Pattern

A dedicated **scratchpad** field in state for intermediate reasoning that the agent
writes to and reads from across steps. Unlike messages (which the LLM always sees),
working memory is a structured side-channel for the agent's internal notes.

**Reference**: Park et al., "Generative Agents" (2023) — agents maintain a
"scratch" memory that tracks plans, observations, and reflections separately
from the conversation stream.

In [ ]:
class WorkingMemoryState(TypedDict):
    """State with explicit working memory scratchpad."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    scratchpad: List[str]  # intermediate notes, not sent to LLM directly
    step_count: int
    max_steps: int


def build_working_memory_agent(tools: list):
    """Agent that maintains a working memory scratchpad."""
    model_with_tools = llm.bind_tools(tools)

    def agent_node(state: WorkingMemoryState) -> dict:
        # Inject scratchpad summary into the system prompt
        scratchpad_text = ""
        if state["scratchpad"]:
            scratchpad_text = (
                "\n\n## Your Working Notes (from previous steps):\n"
                + "\n".join(f"- {note}" for note in state["scratchpad"][-5:])
            )

        system = SystemMessage(
            content=AGENT_SYSTEM_PROMPT + scratchpad_text +
            "\n\nAfter each tool call, mentally note what you learned."
        )
        response = model_with_tools.invoke([system] + list(state["messages"]))

        # Extract a working note from the response
        new_notes = list(state["scratchpad"])
        if response.content and not response.tool_calls:
            new_notes.append(f"Step {state['step_count']}: Produced final answer.")
        elif response.tool_calls:
            tool_names = [tc["name"] for tc in response.tool_calls]
            new_notes.append(f"Step {state['step_count']}: Called {tool_names}")

        return {
            "messages": [response],
            "scratchpad": new_notes,
            "step_count": state["step_count"] + 1,
        }

    tool_node = ToolNode(tools)

    def should_continue(state: WorkingMemoryState) -> Literal["tools", "end"]:
        if state["step_count"] >= state["max_steps"]:
            return "end"
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return "end"

    wf = StateGraph(WorkingMemoryState)
    wf.add_node("agent", agent_node)
    wf.add_node("tools", tool_node)
    wf.add_edge(START, "agent")
    wf.add_conditional_edges(
        "agent", should_continue, {"tools": "tools", "end": END}
    )
    wf.add_edge("tools", "agent")

    return wf.compile(checkpointer=MemorySaver())


wm_agent = build_working_memory_agent(ALL_TOOLS_WITH_RAG)
print("Working Memory Agent compiled.")

# Demo
initial_wm = WorkingMemoryState(
    messages=[HumanMessage(content="What is our remote work policy?")],
    scratchpad=[], step_count=0, max_steps=10,
)
config = {"configurable": {"thread_id": "wm-demo"}}
result = wm_agent.invoke(initial_wm, config)
print("\nScratchpad after run:", result.get("scratchpad", []))

Working Memory Agent compiled.

Scratchpad after run: ["Step 0: Called ['rag_retrieve']", 'Step 1: Produced final answer.']


# 3) Multi-Agent Systems

When a single agent struggles with complex tasks, we can decompose the work across
**multiple specialized agents**, each with its own system prompt, tool access, and
expertise domain.

Three common patterns:
1. **Supervisor**: A coordinator LLM routes work to specialist agents
2. **Debate**: Two agents argue opposing positions; a judge selects the winner
3. **Delegation**: A manager decomposes tasks, delegates to workers, synthesizes results

**Reference**: Wu et al., "AutoGen: Enabling Next-Gen LLM Applications via
Multi-Agent Conversation" (2023).

## 3.1 Supervisor Orchestration

A **supervisor LLM** routes tasks to specialist agents (researcher, writer, reviewer).
Each specialist has its own system prompt and tool access. The supervisor sees the
full state and decides which agent should act next.

```
┌──────────────────────────────────────────────────────────┐
│             Supervisor Multi-Agent System                 │
│                                                          │
│  START ──→ supervisor ──→ [researcher | writer | reviewer]│
│              ↑                      │                    │
│              └──────────────────────┘                    │
│                                                          │
│         supervisor decides FINISH ──→ END                │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
SUPERVISOR_SYSTEM_PROMPT = """You are a workflow supervisor coordinating specialist agents.

## Available Agents
- **researcher**: Gathers information using web search and knowledge base. Use when data is needed.
- **writer**: Synthesizes findings into a polished response. Use after enough data is gathered.
- **reviewer**: Evaluates the writer's output for accuracy. Use after writer produces a draft.
- **FINISH**: End the workflow. Use when reviewer has accepted the output.

## Rules
- Start with researcher (unless query needs no research)
- Only send to writer after researcher has gathered data
- Only send to reviewer after writer has produced a draft
- If reviewer says REVISE, route back to the appropriate agent

Respond with ONLY the next agent name: researcher, writer, reviewer, or FINISH."""


def build_multi_agent_supervisor(tools: list):
    """Supervisor-based multi-agent system."""
    supervisor_llm = ChatOpenAI(model="gpt-4o", temperature=0)
    researcher_llm = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(tools)
    writer_llm = ChatOpenAI(model="gpt-4o", temperature=0)
    reviewer_llm = ChatOpenAI(model="gpt-4o", temperature=0)

    class MultiAgentState(TypedDict):
        messages: Annotated[Sequence[BaseMessage], operator.add]
        next_agent: str
        research_data: List[str]
        draft: Optional[str]
        review: Optional[str]
        step_count: int
        max_steps: int

    tool_node = ToolNode(tools)

    def supervisor_node(state: MultiAgentState) -> dict:
        """LLM-based routing decision."""
        context = (
            f"User query: {state['messages'][0].content}\n"
            f"Research gathered: {len(state['research_data'])} items\n"
            f"Draft written: {'yes' if state.get('draft') else 'no'}\n"
            f"Review: {state.get('review', 'pending')}\n"
            f"Step: {state['step_count']}/{state['max_steps']}"
        )
        response = supervisor_llm.invoke([
            SystemMessage(content=SUPERVISOR_SYSTEM_PROMPT),
            HumanMessage(content=context),
        ])
        decision = response.content.strip().lower()

        if "researcher" in decision:
            next_ag = "researcher"
        elif "writer" in decision:
            next_ag = "writer"
        elif "reviewer" in decision:
            next_ag = "reviewer"
        else:
            next_ag = "FINISH"

        return {"next_agent": next_ag, "step_count": state["step_count"] + 1}

    def researcher_node(state: MultiAgentState) -> dict:
        """Gather information using tools."""
        query = state["messages"][0].content
        response = researcher_llm.invoke([
            SystemMessage(content="You are a thorough researcher. Use tools to gather facts."),
            HumanMessage(content=f"Research: {query}"),
        ])

        new_data = list(state["research_data"])
        if response.tool_calls:
            for tc in response.tool_calls:
                tool_fn = {t.name: t for t in tools}.get(tc["name"])
                if tool_fn:
                    result = tool_fn.invoke(tc["args"])
                    new_data.append(str(result))
        elif response.content:
            new_data.append(response.content)

        return {
            "messages": [AIMessage(content=f"[Researcher] Gathered {len(new_data)} data points.")],
            "research_data": new_data,
        }

    def writer_node(state: MultiAgentState) -> dict:
        """Synthesize research into a polished response."""
        query = state["messages"][0].content
        research = "\n".join(state["research_data"])
        response = writer_llm.invoke([
            SystemMessage(content="You are a skilled writer. Create a clear, well-structured response."),
            HumanMessage(content=f"Query: {query}\n\nResearch data:\n{research}"),
        ])
        return {
            "messages": [AIMessage(content=f"[Writer] {response.content}")],
            "draft": response.content,
        }

    def reviewer_node(state: MultiAgentState) -> dict:
        """Review the draft for quality."""
        response = reviewer_llm.invoke([
            SystemMessage(content=(
                "You are a reviewer. Evaluate the draft for accuracy and completeness. "
                "Respond with 'ACCEPT: reason' or 'REVISE: specific feedback'."
            )),
            HumanMessage(content=f"Draft:\n{state.get('draft', '')}"),
        ])
        return {
            "messages": [AIMessage(content=f"[Reviewer] {response.content}")],
            "review": response.content,
        }

    # Supervisor routes to the next agent
    def route_supervisor(state: MultiAgentState) -> str:
        next_ag = state.get("next_agent", "FINISH")
        if next_ag == "FINISH" or state["step_count"] >= state["max_steps"]:
            return "end"
        return next_ag

    wf = StateGraph(MultiAgentState)
    wf.add_node("supervisor", supervisor_node)
    wf.add_node("researcher", researcher_node)
    wf.add_node("writer", writer_node)
    wf.add_node("reviewer", reviewer_node)

    wf.add_edge(START, "supervisor")
    wf.add_conditional_edges("supervisor", route_supervisor, {
        "researcher": "researcher",
        "writer": "writer",
        "reviewer": "reviewer",
        "end": END,
    })
    # All specialists route back to supervisor
    wf.add_edge("researcher", "supervisor")
    wf.add_edge("writer", "supervisor")
    wf.add_edge("reviewer", "supervisor")
    return wf.compile(checkpointer=MemorySaver())

multi_agent = build_multi_agent_supervisor(ALL_TOOLS_WITH_RAG)
print("Multi-Agent Supervisor compiled. Nodes: supervisor, researcher, writer, reviewer")

Multi-Agent Supervisor compiled. Nodes: supervisor, researcher, writer, reviewer


In [ ]:
# === Test ===
def run_multi_agent(query: str, thread_id: str = "multi-demo"):
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    initial = {
        "messages": [HumanMessage(content=query)],
        "next_agent": "", "research_data": [],
        "draft": None, "review": None,
        "step_count": 0, "max_steps": 10,
    }
    config = {"configurable": {"thread_id": thread_id}}

    for event in multi_agent.stream(initial, config, stream_mode="updates"):
        for node, output in event.items():
            if "next_agent" in output:
                print(f"  SUPERVISOR -> {output['next_agent']}")
            if "messages" in output:
                for msg in output["messages"]:
                    if isinstance(msg, AIMessage) and msg.content:
                        print(f"  [{node}] {msg.content[:150]}")
            if "draft" in output and output["draft"]:
                print(f"  DRAFT: {output['draft'][:200]}...")

    print(f"{'='*60}")

run_multi_agent("Summarize our company's security policy.", "multi-demo-1")


Query: Summarize our company's security policy.
  SUPERVISOR -> researcher
  [researcher] [Researcher] Gathered 1 data points.
  SUPERVISOR -> writer
  [writer] [Writer] Our company's security policy emphasizes the importance of strong authentication and password management. It mandates the use of two-factor a
  DRAFT: Our company's security policy emphasizes the importance of strong authentication and password management. It mandates the use of two-factor authentication (2FA) to enhance security. Passwords must be ...
  SUPERVISOR -> reviewer
  [reviewer] [Reviewer] ACCEPT: The draft accurately and completely outlines the key components of the company's security policy regarding authentication and passw
  SUPERVISOR -> FINISH


## 3.2 Debate Pattern

Two agents argue opposing positions on a question. A judge LLM evaluates the
arguments and selects the stronger position. This pattern improves answer quality
by forcing the model to consider multiple perspectives.

```
START ──→ proponent ──→ opponent ──→ judge ──→ END
              ↑              │
              └── (optional round 2)
```

In [ ]:
def run_debate(question: str, rounds: int = 1) -> str:
    """Simple debate between two LLM agents with a judge."""
    debate_llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
    judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)

    pro_args, con_args = [], []

    for r in range(rounds):
        # Proponent
        pro_context = "\n".join(con_args) if con_args else "No counter-arguments yet."
        pro_resp = debate_llm.invoke([
            SystemMessage(content="You argue IN FAVOR of the proposition. Be concise but rigorous."),
            HumanMessage(content=f"Proposition: {question}\nOpponent's arguments: {pro_context}"),
        ])
        pro_args.append(pro_resp.content)
        print(f"  PRO (round {r+1}): {pro_resp.content[:200]}...")

        # Opponent
        con_resp = debate_llm.invoke([
            SystemMessage(content="You argue AGAINST the proposition. Be concise but rigorous."),
            HumanMessage(content=f"Proposition: {question}\nProponent's arguments: {pro_args[-1]}"),
        ])
        con_args.append(con_resp.content)
        print(f"  CON (round {r+1}): {con_resp.content[:200]}...")

    # Judge evaluates
    judge_resp = judge_llm.invoke([
        SystemMessage(content=(
            "You are an impartial judge. Evaluate both sides and declare a winner "
            "with reasoning. Format: 'WINNER: [PRO/CON]. Reasoning: ...'"
        )),
        HumanMessage(content=(
            f"Proposition: {question}\n\n"
            f"PRO arguments:\n" + "\n".join(pro_args) +
            f"\n\nCON arguments:\n" + "\n".join(con_args)
        )),
    ])
    print(f"\n  JUDGE: {judge_resp.content[:300]}")
    return judge_resp.content


print("Debate: Should AI agents have unrestricted internet access?")
run_debate("AI agents should have unrestricted internet access for maximum capability.", rounds=1)

Debate: Should AI agents have unrestricted internet access?
  PRO (round 1): 1. **Enhanced Learning and Adaptability**: Unrestricted internet access allows AI agents to continuously learn from a vast and diverse set of data sources. This constant influx of information enables ...
  CON (round 1): While unrestricted internet access for AI agents might seem beneficial, there are significant risks and drawbacks that must be considered, making it a questionable proposition.

1. **Security Risks**:...

  JUDGE: WINNER: CON. Reasoning: While the PRO side presents compelling arguments about the potential benefits of unrestricted internet access for AI agents, such as enhanced learning, comprehensive problem-solving, and real-time responsiveness, the CON side effectively highlights significant risks and drawb


'WINNER: CON. Reasoning: While the PRO side presents compelling arguments about the potential benefits of unrestricted internet access for AI agents, such as enhanced learning, comprehensive problem-solving, and real-time responsiveness, the CON side effectively highlights significant risks and drawbacks that outweigh these benefits. The security risks, data privacy concerns, and potential for misinformation and bias are particularly concerning, as they could lead to harmful consequences in critical sectors like healthcare and finance. Additionally, the lack of accountability and ethical concerns raised by the CON side underscore the need for a more controlled and monitored approach to AI internet access. These issues present substantial challenges that cannot be overlooked, making the CON argument more persuasive in advocating for a balanced approach that mitigates risks while still allowing AI to be effective.'

## 3.3 Delegation Pattern

A **manager** decomposes the task into independent subtasks, delegates each to a
**worker** agent, collects results, and synthesizes a final answer. This is similar
to MapReduce for LLM workflows.

```
Manager: "Break task into subtasks"
  ├── Worker 1: subtask A → result A
  ├── Worker 2: subtask B → result B
  └── Worker 3: subtask C → result C
Manager: "Synthesize results A + B + C → final answer"
```

In [ ]:
def delegation_workflow(query: str) -> str:
    """Manager-worker delegation pattern."""
    manager_llm = ChatOpenAI(model="gpt-4o", temperature=0)
    worker_llm = ChatOpenAI(model="gpt-4o", temperature=0)

    # Step 1: Manager decomposes
    decomp = manager_llm.invoke([
        SystemMessage(content=(
            "Decompose the query into 2-4 independent subtasks. "
            "Output each subtask on a separate line, prefixed with a number."
        )),
        HumanMessage(content=query),
    ])
    subtasks = [s.strip() for s in decomp.content.strip().split("\n") if s.strip()]
    print(f"  Manager decomposed into {len(subtasks)} subtasks.\nSub-tasks:\n{subtasks}")

    # Step 2: Workers handle each subtask
    results = []
    for i, task in enumerate(subtasks):
        resp = worker_llm.invoke([
            SystemMessage(content="Answer the given subtask concisely."),
            HumanMessage(content=task),
        ])
        results.append(f"Subtask {i+1}: {resp.content[:200]}")
        print(f"  Worker {i+1}: {resp.content[:100]}...")

    # Step 3: Manager synthesizes
    synthesis = manager_llm.invoke([
        SystemMessage(content="Synthesize the worker results into a coherent final answer."),
        HumanMessage(content=f"Query: {query}\n\nResults:\n" + "\n".join(results)),
    ])
    print(f"\n  FINAL: {synthesis.content[:300]}")
    return synthesis.content


print("Delegation demo:")
delegation_workflow("Compare Python, Rust, and Go for building AI agent backends.")

Delegation demo:
  Manager decomposed into 4 subtasks.
Sub-tasks:
['1. Analyze the strengths and weaknesses of Python for building AI agent backends, focusing on its libraries, performance, and ease of use.', '2. Evaluate the capabilities of Rust for AI agent backend development, considering its performance, safety features, and ecosystem support.', '3. Assess the suitability of Go for creating AI agent backends, taking into account its concurrency model, performance, and library support.', '4. Compare and contrast the findings from the analyses of Python, Rust, and Go to determine which language is most suitable for building AI agent backends based on specific criteria such as performance, ease of development, and ecosystem support.']
  Worker 1: **Strengths:**

1. **Rich Libraries and Frameworks**: Python boasts a vast ecosystem of libraries an...
  Worker 2: Rust offers several capabilities that make it suitable for AI agent backend development:

1. **Perfo...
  Worker 3: Go is a su

"When comparing Python, Rust, and Go for building AI agent backends, each language offers unique strengths and weaknesses that cater to different aspects of development:\n\n1. **Python**:\n   - **Strengths**:\n     - **Rich Libraries and Frameworks**: Python has a vast ecosystem of libraries and frameworks such as TensorFlow, PyTorch, scikit-learn, and Keras, which are specifically designed for AI and machine learning tasks. This makes Python an excellent choice for rapid prototyping and development of AI models.\n     - **Ease of Use**: Python's syntax is simple and readable, which makes it accessible for developers and researchers who may not have a strong programming background.\n     - **Community and Support**: Python has a large and active community, providing extensive documentation, tutorials, and support for AI development.\n   - **Weaknesses**:\n     - **Performance**: Python is generally slower than compiled languages like Rust and Go, which can be a limitation for performan

# 4) External Tool

## 4.1 MCP (Model Context Protocol)

**MCP** is a standardized protocol (by Anthropic, 2024) for communication between
LLM applications and external tools/resources.

It replaces ad-hoc tool integrations
with a universal, discoverable, bidirectional protocol.

```
┌──────────────────────────────────────────────────────────────────┐
│                    MCP Architecture                              │
│                                                                  │
│  ┌──────────┐     ┌──────────┐     ┌───────────────────────┐   │
│  │   Host   │────→│  Client  │────→│       Server          │   │
│  │ (IDE/App)│     │ (in-app) │     │  (tools/resources)    │   │
│  └──────────┘     └──────────┘     │                       │   │
│                                    │  - tools (callable)   │   │
│  Transport: stdio or HTTP+SSE      │  - resources (data)   │   │
│  Format: JSON-RPC 2.0              │  - prompts (templates)│   │
│                                    │  - sampling (LLM call)│   │
│                                    └───────────────────────┘   │
└──────────────────────────────────────────────────────────────────┘
```

### How MCP differs from simple function calling:
| Feature              | Function Calling       | MCP                     |
|----------------------|------------------------|-------------------------|
| Direction            | Unidirectional (LLM→)  | Bidirectional           |
| Discovery            | Static schema          | Dynamic (tools/list)    |
| Protocol             | Vendor-specific        | Standardized JSON-RPC   |
| Resources            | Not supported          | First-class (URIs)      |
| Composability        | Per-integration        | Any MCP client ↔ server |

### Key MCP concepts:
- **Tools**: Callable functions (like function calling, but discoverable)
- **Resources**: Data sources the LLM can read (files, DB rows, API responses)
- **Prompts**: Reusable prompt templates the server provides
- **Sampling**: Server can request LLM completions from the host (bidirectional!)

**Reference**: Anthropic, "Model Context Protocol" specification (2024).

```python
# Server Side
from mcp.server.fastmcp import FastMCP
import re
mcp_server = FastMCP("Company_Tools_Server")

@mcp_server.tool()
def mcp_calculator(expression: str) -> str:
    '''Evaluate a mathematical expression. (MCP Tool)
    Use for any computation: arithmetic, percentages, comparisons.
    '''
    sanitized = re.sub(r'[^0-9+\-*/().,%\s]', '', expression)
    try:
        return f"{sanitized.strip()} = {eval(sanitized)}"
    except Exception as e:
        return f"Error: {e}"

if __name__ == "__main__":
    # Run as a web service over the network instead of a local subprocess
    mcp_server.run(transport='sse', host='0.0.0.0', port=8000)
```

```python
# Client Side
import asyncio
from mcp.client.sse import sse_client
from mcp.client.session import ClientSession
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import json

async def run_mcp_with_llm():
    print("Connecting to Remote MCP Server...")

    mcp_server_url = "http://localhost:8000/sse"
    
    async with sse_client(mcp_server_url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            print("\n[SUCCESS] Connected to MCP Server via SSE")

            # A. Discover tools from MCP Server
            mcp_tools = await session.list_tools()
            print("\n[SEARCH] Discovered Tools from MCP:")
            for t in mcp_tools.tools:
                print(f"  - {t.name}: {t.description}")

            # B. Format MCP tools for OpenAI's function calling
            openai_tools = []
            for t in mcp_tools.tools:
                openai_tools.append({
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema
                    }
                })

            # C. Ask the LLM a question
            llm = ChatOpenAI(model="gpt-4o")
            query = "What is 1547 multiplied by 23?"
            print(f"\n[USER] User Query: '{query}'")

            response = llm.invoke(
                [HumanMessage(content=query)],
                tools=openai_tools
            )
            print(f"\n[LLM] LLM decided to call: {response.tool_calls}")

            # D. Execute the tool dynamically via MCP
            if response.tool_calls:
                for tc in response.tool_calls:
                    print(
                        f"\n[EXEC] Executing '{tc['name']}' via MCP with args: {tc['args']}"
                    )
                    # Call the MCP server
                    mcp_result = await session.call_tool(tc["name"], tc["args"])

                    # Extract the result text
                    result_text = mcp_result.content[0].text
                    print(f"\n[MCP] MCP Server Result: {result_text}")

                    # E. Provide result back to LLM for final answer
                    final_response = llm.invoke([
                        HumanMessage(content=query),
                        response,
                        {
                            "role": "tool",
                            "tool_call_id": tc["id"],
                            "name": tc["name"],
                            "content": result_text
                        }
                    ])
                    print(
                        f"\n[FINAL] Final LLM Answer: {final_response.content}"
                    )

if __name__ == "__main__":
    asyncio.run(run_mcp_with_llm())
```

Output:

```
Executing the client script in a native OS subprocess...
------------------------------------------------------------
Starting MCP Server as a subprocess...

[SUCCESS] Connected to MCP Server via stdio

[SEARCH] Discovered Tools from MCP:
  - mcp_calculator: Evaluate a mathematical expression. (MCP Tool)
    Use for any computation: arithmetic, percentages, comparisons.
    

[USER] User Query: 'What is 1547 multiplied by 23?'

[LLM] LLM decided to call: [{'name': 'mcp_calculator', 'args': {'expression': '1547 * 23'}, 'id': 'call_UvoPBVvPWHVoCeybV12LHmEg', 'type': 'tool_call'}]

[EXEC] Executing 'mcp_calculator' via MCP with args: {'expression': '1547 * 23'}

[MCP] MCP Server Result: 1547 * 23 = 35581

[FINAL] Final LLM Answer: The result of multiplying 1547 by 23 is 35,581.
```

## 4.2 Sandboxing and Security for Code Execution

LLM-generated code can be **dangerous** — it may attempt file deletion, network
access, or infinite loops. Any agent that executes code must do so in a **sandbox**.

### Approaches:
| Method          | Isolation Level | Overhead  | Use Case                    |
|-----------------|----------------|-----------|-----------------------------|
| subprocess      | Low (OS-level)  | Minimal   | Quick prototyping           |
| Docker          | Medium          | Moderate  | Production, multi-language  |
| gVisor          | High            | Moderate  | Security-critical           |
| Firecracker     | Very High       | Higher    | Multi-tenant production     |
| E2B / Modal     | High (cloud)    | Network   | Serverless execution        |

### Security checklist:
- No network access from sandbox
- No file system writes outside temp directory
- CPU and memory resource limits
- Hard timeout on execution
- No access to environment variables (API keys!)

**Reference**: Best practices for LLM code execution — OWASP LLM Top 10 (2023).

In [ ]:
import subprocess
import tempfile

@tool
def sandboxed_python_exec(code: str) -> str:
    """Execute Python code in a sandboxed subprocess with timeout.
    Use for calculations, data processing, or any computation.
    The code runs in an isolated process with no network access.

    Args:
        code: Python code to execute. Must print results to stdout.
    """
    # Security: block dangerous imports and operations
    blocked_patterns = [
        "import os", "import sys", "import subprocess",
        "import socket", "import requests", "import urllib",
        "open(", "__import__", "eval(", "exec(",
        "shutil", "pathlib", "glob",
    ]
    for pattern in blocked_patterns:
        if pattern in code:
            return f"BLOCKED: Code contains forbidden pattern '{pattern}'. "\
                   f"Only pure computation is allowed."

    try:
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".py", delete=True
        ) as f:
            f.write(code)
            f.flush()

            result = subprocess.run(
                ["python3", f.name],
                capture_output=True,
                text=True,
                timeout=10,  # Hard 10-second timeout
                env={"PATH": "/usr/bin:/usr/local/bin"},  # Minimal env, no API keys
            )

            if result.returncode == 0:
                return result.stdout.strip() or "(no output)"
            return f"Error (exit {result.returncode}): {result.stderr.strip()[:500]}"

    except subprocess.TimeoutExpired:
        return "TIMEOUT: Code execution exceeded 10-second limit."
    except Exception as e:
        return f"Sandbox error: {e}"


# Demo
print(sandboxed_python_exec.invoke({"code": "print(sum(range(100)))"}))
print(sandboxed_python_exec.invoke({"code": "import os; os.listdir('/')"}))

4950
BLOCKED: Code contains forbidden pattern 'import os'. Only pure computation is allowed.


## 4.3 Guardrails and Human-in-the-Loop

Guardrails protect the agent from producing harmful, incorrect, or expensive outputs.
LangGraph supports **human-in-the-loop** via `interrupt_before` — the graph pauses
before executing a specified node and waits for human approval.

### Guardrail layers:
```
┌────────────────────────────────────────────────────────┐
│                  Guardrail Stack                        │
│                                                        │
│  1. Step limit         max_steps in state              │
│  2. Token budget       tiktoken counting               │
│  3. PII detection      regex on input/output           │
│  4. Output validation  format/content checks           │
│  5. Human-in-the-loop  interrupt_before=["tools"]      │
└────────────────────────────────────────────────────────┘
```

**Reference**: NeMo Guardrails (NVIDIA, 2023); Guardrails AI.

In [ ]:
import tiktoken as _tiktoken

# PII patterns for guardrail detection
PII_PATTERNS = {
    "email": r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
    "phone": r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
    "ssn": r'\b\d{3}-\d{2}-\d{4}\b',
    "credit_card": r'\b\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}\b',
}


def detect_pii(text: str) -> List[str]:
    """Scan text for PII patterns. Returns list of detected PII types."""
    found = []
    for pii_type, pattern in PII_PATTERNS.items():
        if re.search(pattern, text):
            found.append(pii_type)
    return found


def build_guarded_agent(tools: list, max_steps: int = 10, max_tokens: int = 50_000):
    """Agent with step limits, token budget, PII detection, and output validation."""
    model_with_tools = llm.bind_tools(tools)
    enc = _tiktoken.encoding_for_model("gpt-4o")

    class GuardedState(TypedDict):
        messages: Annotated[Sequence[BaseMessage], operator.add]
        step_count: int
        max_steps: int
        total_tokens: int
        max_tokens: int
        guardrail_violations: List[str]

    def input_guard(state: GuardedState) -> dict:
        """Check the user query for PII before processing."""
        query = state["messages"][0].content if state["messages"] else ""
        pii = detect_pii(query)
        violations = list(state.get("guardrail_violations", []))
        if pii:
            violations.append(f"INPUT_PII: {pii}")
            return {
                "messages": [AIMessage(content=(
                    f"I detected potential PII ({', '.join(pii)}) in your query. "
                    f"Please remove personal information and try again."
                ))],
                "guardrail_violations": violations,
            }
        return {"guardrail_violations": violations}

    def agent_node(state: GuardedState) -> dict:
        system = SystemMessage(content=AGENT_SYSTEM_PROMPT)
        response = model_with_tools.invoke([system] + list(state["messages"]))
        tokens_used = len(enc.encode(response.content or ""))
        return {
            "messages": [response],
            "step_count": state["step_count"] + 1,
            "total_tokens": state["total_tokens"] + tokens_used,
        }

    tool_node = ToolNode(tools)

    def output_guard(state: GuardedState) -> dict:
        """Validate the final output: check for PII and format."""
        last = state["messages"][-1]
        content = last.content if hasattr(last, "content") else ""
        violations = list(state.get("guardrail_violations", []))

        pii = detect_pii(content)
        if pii:
            violations.append(f"OUTPUT_PII: {pii}")
            return {
                "messages": [AIMessage(content="[Output redacted due to PII detection]")],
                "guardrail_violations": violations,
            }
        return {"guardrail_violations": violations}

    # Routing
    def after_input_guard(state: GuardedState) -> Literal["agent", "end"]:
        violations = state.get("guardrail_violations", [])
        if any("INPUT_PII" in v for v in violations):
            return "end"
        return "agent"

    def after_agent(state: GuardedState) -> Literal["tools", "output_guard"]:
        if state["step_count"] >= state["max_steps"]:
            return "output_guard"
        if state["total_tokens"] >= state["max_tokens"]:
            return "output_guard"
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return "output_guard"

    wf = StateGraph(GuardedState)
    wf.add_node("input_guard", input_guard)
    wf.add_node("agent", agent_node)
    wf.add_node("tools", tool_node)
    wf.add_node("output_guard", output_guard)

    wf.add_edge(START, "input_guard")
    wf.add_conditional_edges(
        "input_guard", after_input_guard, {"agent": "agent", "end": END}
    )
    wf.add_conditional_edges(
        "agent", after_agent, {"tools": "tools", "output_guard": "output_guard"}
    )
    wf.add_edge("tools", "agent")
    wf.add_edge("output_guard", END)

    return wf.compile(checkpointer=MemorySaver())


guarded_agent = build_guarded_agent(ALL_TOOLS_WITH_RAG)
print("Guarded Agent compiled. Nodes: input_guard, agent, tools, output_guard")

Guarded Agent compiled. Nodes: input_guard, agent, tools, output_guard


In [ ]:
# Demo: PII detection guardrail
print("--- Normal query ---")
normal_state = {
    "messages": [HumanMessage(content="What is 2+2?")],
    "step_count": 0, "max_steps": 10,
    "total_tokens": 0, "max_tokens": 50_000,
    "guardrail_violations": [],
}
result = guarded_agent.invoke(normal_state, {"configurable": {"thread_id": "guard-1"}})
print(f"Violations: {result.get('guardrail_violations', [])}")

print("\n--- Query with PII ---")
pii_state = {
    "messages": [HumanMessage(content="My SSN is 123-45-6789. What should I do?")],
    "step_count": 0, "max_steps": 10,
    "total_tokens": 0, "max_tokens": 50_000,
    "guardrail_violations": [],
}
result = guarded_agent.invoke(pii_state, {"configurable": {"thread_id": "guard-2"}})
print(f"Violations: {result.get('guardrail_violations', [])}")
for m in result["messages"]:
    if isinstance(m, AIMessage):
        print(f"Response: {m.content}")

--- Normal query ---
Violations: []

--- Query with PII ---
Violations: ["INPUT_PII: ['ssn']"]
Response: I detected potential PII (ssn) in your query. Please remove personal information and try again.


In [ ]:
# Human-in-the-loop: interrupt_before pauses the graph before tool execution
# The user must approve before tools run.

hitl_agent = build_react_agent(ALL_TOOLS_WITH_RAG)

# To enable HITL, recompile with interrupt_before
model_with_tools = llm.bind_tools(ALL_TOOLS_WITH_RAG)

def agent_node_hitl(state: AgentState) -> dict:
    system = SystemMessage(content=AGENT_SYSTEM_PROMPT)
    response = model_with_tools.invoke([system] + list(state["messages"]))
    return {"messages": [response], "step_count": state["step_count"] + 1}

def should_continue_hitl(state: AgentState) -> Literal["tools", "end"]:
    if state["step_count"] >= state["max_steps"]:
        return "end"
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"

hitl_wf = StateGraph(AgentState)
hitl_wf.add_node("agent", agent_node_hitl)
hitl_wf.add_node("tools", ToolNode(ALL_TOOLS_WITH_RAG))
hitl_wf.add_edge(START, "agent")
hitl_wf.add_conditional_edges(
    "agent", should_continue_hitl, {"tools": "tools", "end": END}
)
hitl_wf.add_edge("tools", "agent")

# interrupt_before=["tools"] means the graph PAUSES before executing tools
# In a production UI, this is where you show the user what tool call the agent
# wants to make and ask for approval before proceeding.
hitl_graph = hitl_wf.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["tools"],
)
print("Human-in-the-loop agent compiled with interrupt_before=['tools']")
print("In production, the graph pauses before tool execution for human approval.")
print("Resume with: graph.invoke(None, config) after approval.")

Human-in-the-loop agent compiled with interrupt_before=['tools']
In production, the graph pauses before tool execution for human approval.
Resume with: graph.invoke(None, config) after approval.


# 5) Complete Production Workflow

This section wires all building blocks into a single production-grade graph:

```
┌──────────────────────────────────────────────────────────────────┐
│              Complete Production Agent Graph                      │
│                                                                  │
│  START ──→ input_guard ──→ planner ──→ agent ──→ [tools?]       │
│                │ PII        │             ↑         │           │
│                ↓            │             └─────────┘           │
│               END           │                  ↓                │
│                             │            reflector              │
│                             │          [ACCEPT?]               │
│                             │        yes │  no                  │
│                             │            ↓   └──→ agent        │
│                             │       output_guard               │
│                             │            │                      │
│                             │            ↓                      │
│                             │           END                     │
└──────────────────────────────────────────────────────────────────┘
```

### Production checklist:
- Input guardrails (PII, injection detection)
- Planning phase for complex queries
- Tool execution with retry logic
- Reflection/self-critique before delivery
- Output guardrails (PII, format validation)
- Observability via LangSmith
- Checkpointing for fault tolerance
- Step and token budget limits

In [ ]:
class ProductionState(TypedDict):
    """Full production agent state with all guardrail fields."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    plan: List[str]
    step_count: int
    max_steps: int
    reflection_notes: List[str]
    total_tokens: int
    max_tokens: int
    guardrail_violations: List[str]
    status: str


def build_production_agent(tools: list):
    """Full production agent: input_guard -> planner -> agent <-> tools -> reflector -> output_guard."""
    model_with_tools = llm.bind_tools(tools)
    planner_llm = ChatOpenAI(model="gpt-4o", temperature=0)
    reflection_llm = ChatOpenAI(model="gpt-4o", temperature=0)
    enc = _tiktoken.encoding_for_model("gpt-4o")

    def input_guard(state: ProductionState) -> dict:
        query = state["messages"][0].content if state["messages"] else ""
        pii = detect_pii(query)
        violations = list(state.get("guardrail_violations", []))
        if pii:
            violations.append(f"INPUT_PII: {pii}")
            return {
                "messages": [AIMessage(content="PII detected. Please remove personal information.")],
                "guardrail_violations": violations,
                "status": "blocked",
            }
        return {"guardrail_violations": violations, "status": "planning"}

    def planner_node(state: ProductionState) -> dict:
        query = state["messages"][0].content
        response = planner_llm.invoke([
            SystemMessage(content=(
                "Create a brief 1-3 step plan to answer this query. "
                "Output numbered steps only."
            )),
            HumanMessage(content=query),
        ])
        steps = [s.strip() for s in response.content.strip().split("\n") if s.strip()]
        return {
            "plan": steps,
            "messages": [AIMessage(content=f"Plan: {len(steps)} steps")],
            "status": "executing",
        }

    def agent_node(state: ProductionState) -> dict:
        system = SystemMessage(content=AGENT_SYSTEM_PROMPT)
        response = model_with_tools.invoke([system] + list(state["messages"]))
        tokens = len(enc.encode(response.content or ""))
        return {
            "messages": [response],
            "step_count": state["step_count"] + 1,
            "total_tokens": state["total_tokens"] + tokens,
        }

    tool_node = ToolNode(tools)

    def reflector_node(state: ProductionState) -> dict:
        query = state["messages"][0].content
        last_answer = ""
        for m in reversed(state["messages"]):
            if isinstance(m, AIMessage) and m.content and not getattr(m, "tool_calls", None):
                last_answer = m.content
                break
        tool_results = [m.content for m in state["messages"] if isinstance(m, ToolMessage)]
        context = (
            f"Question: {query}\n\nTool results:\n"
            + "\n".join(f"- {r[:200]}" for r in tool_results[-5:])
            + f"\n\nAnswer:\n{last_answer}"
        )
        verdict = reflection_llm.invoke([
            SystemMessage(content=REFLECTION_SYSTEM_PROMPT),
            HumanMessage(content=context),
        ])
        note = verdict.content.strip()
        new_notes = list(state["reflection_notes"]) + [note]
        if "REVISE" in note.upper() and len(new_notes) <= 2:
            return {
                "reflection_notes": new_notes,
                "messages": [HumanMessage(content=f"[REVISION] {note}")],
                "status": "reflecting",
            }
        return {"reflection_notes": new_notes, "status": "complete"}

    def output_guard(state: ProductionState) -> dict:
        last = state["messages"][-1]
        content = last.content if hasattr(last, "content") else ""
        violations = list(state.get("guardrail_violations", []))
        pii = detect_pii(content)
        if pii:
            violations.append(f"OUTPUT_PII: {pii}")
            return {
                "messages": [AIMessage(content="[Redacted]")],
                "guardrail_violations": violations,
            }
        return {"guardrail_violations": violations}

    # Routing functions
    def after_input(state: ProductionState) -> Literal["planner", "end"]:
        if state.get("status") == "blocked":
            return "end"
        return "planner"

    def after_agent(state: ProductionState) -> Literal["tools", "reflector"]:
        if state["step_count"] >= state["max_steps"]:
            return "reflector"
        if state["total_tokens"] >= state["max_tokens"]:
            return "reflector"
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return "reflector"

    def after_reflector(state: ProductionState) -> Literal["agent", "output_guard"]:
        if state.get("status") == "complete":
            return "output_guard"
        return "agent"

    wf = StateGraph(ProductionState)
    wf.add_node("input_guard", input_guard)
    wf.add_node("planner", planner_node)
    wf.add_node("agent", agent_node)
    wf.add_node("tools", tool_node)
    wf.add_node("reflector", reflector_node)
    wf.add_node("output_guard", output_guard)

    wf.add_edge(START, "input_guard")
    wf.add_conditional_edges(
        "input_guard", after_input, {"planner": "planner", "end": END}
    )
    wf.add_edge("planner", "agent")
    wf.add_conditional_edges(
        "agent", after_agent, {"tools": "tools", "reflector": "reflector"}
    )
    wf.add_edge("tools", "agent")
    wf.add_conditional_edges(
        "reflector", after_reflector, {"agent": "agent", "output_guard": "output_guard"}
    )
    wf.add_edge("output_guard", END)

    return wf.compile(checkpointer=MemorySaver())


production_agent = build_production_agent(ALL_TOOLS_WITH_RAG)
print("Production Agent compiled.")
print("Graph: input_guard -> planner -> agent <-> tools -> reflector -> output_guard -> END")

Production Agent compiled.
Graph: input_guard -> planner -> agent <-> tools -> reflector -> output_guard -> END


In [ ]:
# Demo: Run the full production agent
def run_production(query: str, thread_id: str):
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    initial = ProductionState(
        messages=[HumanMessage(content=query)],
        plan=[], step_count=0, max_steps=10,
        reflection_notes=[], total_tokens=0, max_tokens=50_000,
        guardrail_violations=[], status="",
    )
    config = {"configurable": {"thread_id": thread_id}}

    for event in production_agent.stream(initial, config, stream_mode="updates"):
        for node, output in event.items():
            if "messages" in output:
                for msg in output["messages"]:
                    if isinstance(msg, AIMessage):
                        if msg.tool_calls:
                            for tc in msg.tool_calls:
                                print(f"  TOOL [{node}] {tc['name']}")
                        elif msg.content:
                            print(f"  [{node}] {msg.content[:200]}")
                    elif isinstance(msg, ToolMessage):
                        print(f"  RESULT [{node}] {msg.content[:100]}...")
            if "status" in output:
                print(f"  STATUS: {output['status']}")

    print(f"{'='*60}")


# Query 1: Factual with tool use
run_production("What is the capital of France?", "prod-1")

# Query 2: Internal policy (RAG)
run_production("How many PTO days do employees get?", "prod-2")

# Query 3: PII blocked
run_production("My SSN is 123-45-6789, look up my benefits.", "prod-3")


Query: What is the capital of France?
  STATUS: planning
  [planner] Plan: 2 steps
  STATUS: executing
  [agent] The capital of France is Paris.
  STATUS: complete

Query: How many PTO days do employees get?
  STATUS: planning
  [planner] Plan: 3 steps
  STATUS: executing
  TOOL [agent] rag_retrieve
  RESULT [tools] Knowledge Base Results:

[distance: 0.412] Vacation Policy: Full-time employees receive 20 days PTO ...
  [agent] Full-time employees receive 20 days of PTO (Paid Time Off) per year, accruing at a rate of 1.67 days per month. Additionally, employees can carry over up to 5 unused PTO days to the next year.
  STATUS: complete

Query: My SSN is 123-45-6789, look up my benefits.
  [input_guard] PII detected. Please remove personal information.
  STATUS: blocked


# Summary

## Building Blocks Table

| # | Building Block        | Key Concept                          | LangGraph Mechanism           |
|---|----------------------|--------------------------------------|-------------------------------|
| 1 | State Design         | TypedDict + reducer pattern          | `Annotated[..., operator.add]`|
| 2 | Tool Definitions     | @tool → JSON schema → function call  | `ToolNode`, `bind_tools()`    |
| 3 | RAG                  | Vector search as an agent tool       | ChromaDB + `@tool`            |
| 4 | ReAct                | Reason + Act cycle                   | Conditional edges + cycles    |
| 5 | Plan-and-Execute     | Plan first, execute with tools       | Multi-node graph              |
| 6 | Reflection           | Self-critique loop                   | Reflector node + feedback     |
| 7 | Short-Term Memory    | Messages in state                    | Reducer accumulation          |
| 8 | Long-Term Memory     | Checkpointing across invocations     | `MemorySaver` / thread_id     |
| 9 | Working Memory       | Scratchpad for intermediate notes    | Dedicated state field         |
| 10| Multi-Agent          | Supervisor routes to specialists     | Conditional edges + sub-nodes |
| 11| Debate               | Opposing agents + judge              | Sequential LLM calls          |
| 12| Delegation           | Manager decomposes, workers execute  | MapReduce pattern             |
| 13| Error Handling       | Retry + backoff + LLM self-recovery  | Custom tool node              |
| 14| MCP                  | Standardized tool protocol           | JSON-RPC 2.0                  |
| 15| Sandboxing           | Isolated code execution              | subprocess + blocklist        |
| 16| Guardrails           | PII, token budget, step limits       | Guard nodes in graph          |
| 17| Human-in-the-Loop    | Pause for approval                   | `interrupt_before`            |

## Key Papers

1. Yao et al., "ReAct: Synergizing Reasoning and Acting in Language Models" (ICLR 2023)
2. Shinn et al., "Reflexion: Language Agents with Verbal Reinforcement Learning" (NeurIPS 2023)
3. Wang et al., "Plan-and-Solve Prompting" (ACL 2023)
4. Wu et al., "AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation" (2023)
5. Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" (NeurIPS 2020)
6. Park et al., "Generative Agents: Interactive Simulacra of Human Behavior" (2023)
7. Anthropic, "Model Context Protocol" specification (2024)

## Production Next Steps

- **Persistence**: Replace `MemorySaver` with `PostgresSaver` for durable state
- **Deployment**: LangGraph Cloud or LangServe for API serving
- **Evaluation**: LangSmith datasets + automated evaluation pipelines
- **Streaming**: Enable `streaming=True` on ChatOpenAI for real-time UI
- **Observability**: Set up LangSmith alerts for latency spikes and error rates
- **Testing**: Unit test each node function independently; integration test the graph
- **Security**: Add proper sandboxing (Docker/E2B) for code execution tools
- **Scale**: Use async (`astream`) for concurrent agent execution